In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm

## Leitura dos dados

In [2]:
embeddings = pd.read_parquet("../../data/datasets/embeddings.parquet")

In [3]:
embedding_cols = [
    'embedding__codefuse_ai__F2LLM_v2_14B__texto',
       'embedding__codefuse_ai__F2LLM_v2_14B__texto_preprocessado',
       'embedding__codefuse_ai__F2LLM_v2_14B__texto_preprocessado_sem_justificativa',
       'embedding__Octen__Octen_Embedding_8B__texto',
       'embedding__Octen__Octen_Embedding_8B__texto_preprocessado',
       'embedding__Octen__Octen_Embedding_8B__texto_preprocessado_sem_justificativa',
       'embedding__Qwen__Qwen3_Embedding_8B__texto',
       'embedding__Qwen__Qwen3_Embedding_8B__texto_preprocessado',
       'embedding__Qwen__Qwen3_Embedding_8B__texto_preprocessado_sem_justificativa',
       'embedding__nvidia__llama_embed_nemotron_8b__texto',
       'embedding__nvidia__llama_embed_nemotron_8b__texto_preprocessado',
       'embedding__nvidia__llama_embed_nemotron_8b__texto_preprocessado_sem_justificativa',
       'embedding__jinaai__jina_embeddings_v5_text_small__clustering__texto',
       'embedding__jinaai__jina_embeddings_v5_text_small__clustering__texto_preprocessado',
       'embedding__jinaai__jina_embeddings_v5_text_small__clustering__texto_preprocessado_sem_justificativa',
       'embedding__jinaai__jina_embeddings_v5_text_small__text_matching__texto',
       'embedding__jinaai__jina_embeddings_v5_text_small__text_matching__texto_preprocessado',
       'embedding__jinaai__jina_embeddings_v5_text_small__text_matching__texto_preprocessado_sem_justificativa',
       'embedding__joaorobson__harrier_oss_v1_27b__texto',
       'embedding__joaorobson__harrier_oss_v1_27b__texto_preprocessado',
       'embedding__joaorobson__harrier_oss_v1_27b__texto_preprocessado_sem_justificativa',
       'embedding__joaorobson__KaLM_Embedding_Gemma3_12B_2511__texto',
       'embedding__joaorobson__KaLM_Embedding_Gemma3_12B_2511__texto_preprocessado',
       'embedding__joaorobson__KaLM_Embedding_Gemma3_12B_2511__texto_preprocessado_sem_justificativa',
       'embedding__openai__text_embedding_3_large__texto',
       'embedding__openai__text_embedding_3_large__texto_preprocessado',
       'embedding__openai__text_embedding_3_large__texto_preprocessado_sem_justificativa',
       'embedding__PORTULAN__serafim_335m_portuguese_pt_sentence_encoder__trunc128__texto',
       'embedding__PORTULAN__serafim_335m_portuguese_pt_sentence_encoder__trunc128__texto_preprocessado',
       'embedding__PORTULAN__serafim_335m_portuguese_pt_sentence_encoder__trunc128__texto_preprocessado_sem_justificativa',
       'embedding__PORTULAN__serafim_335m_portuguese_pt_sentence_encoder__meanpool128__texto',
       'embedding__PORTULAN__serafim_335m_portuguese_pt_sentence_encoder__meanpool128__texto_preprocessado',
       'embedding__PORTULAN__serafim_335m_portuguese_pt_sentence_encoder__meanpool128__texto_preprocessado_sem_justificativa',
       'embedding__PORTULAN__serafim_335m_portuguese_pt_sentence_encoder_ir__trunc128__texto',
       'embedding__PORTULAN__serafim_335m_portuguese_pt_sentence_encoder_ir__trunc128__texto_preprocessado',
       'embedding__PORTULAN__serafim_335m_portuguese_pt_sentence_encoder_ir__trunc128__texto_preprocessado_sem_justificativa',
       'embedding__PORTULAN__serafim_335m_portuguese_pt_sentence_encoder_ir__meanpool128__texto',
       'embedding__PORTULAN__serafim_335m_portuguese_pt_sentence_encoder_ir__meanpool128__texto_preprocessado',
       'embedding__PORTULAN__serafim_335m_portuguese_pt_sentence_encoder_ir__meanpool128__texto_preprocessado_sem_justificativa',
       'embedding__josedossantos__bertimbau_tuned__trunc512__texto',
       'embedding__josedossantos__bertimbau_tuned__trunc512__texto_preprocessado',
       'embedding__josedossantos__bertimbau_tuned__trunc512__texto_preprocessado_sem_justificativa',
       'embedding__josedossantos__bertimbau_tuned__meanpool512__texto',
       'embedding__josedossantos__bertimbau_tuned__meanpool512__texto_preprocessado',
       'embedding__josedossantos__bertimbau_tuned__meanpool512__texto_preprocessado_sem_justificativa',
       'embedding__ICT_TIME_and_Querit__ICT_TIME_and_Querit_embedding_v1__texto',
       'embedding__ICT_TIME_and_Querit__ICT_TIME_and_Querit_embedding_v1__texto_preprocessado',
       'embedding__ICT_TIME_and_Querit__ICT_TIME_and_Querit_embedding_v1__texto_preprocessado_sem_justificativa',
       'embedding__gemini__gemini_embedding_2__raw__texto',
       'embedding__gemini__gemini_embedding_2__raw__texto_preprocessado',
       'embedding__gemini__gemini_embedding_2__raw__texto_preprocessado_sem_justificativa',
       'embedding__gemini__gemini_embedding_2__clustering__texto',
       'embedding__gemini__gemini_embedding_2__clustering__texto_preprocessado',
       'embedding__gemini__gemini_embedding_2__clustering__texto_preprocessado_sem_justificativa',
       'embedding__gemini__gemini_embedding_2__sts__texto',
       'embedding__gemini__gemini_embedding_2__sts__texto_preprocessado',
       'embedding__gemini__gemini_embedding_2__sts__texto_preprocessado_sem_justificativa']

In [5]:
for materia in embeddings.materia.unique():
    df_count_temas = embeddings[embeddings.materia == materia]
    total_amostras_materia = len(df_count_temas)  # Total de amostras (linhas) desta matéria
        
    print(f"\n--- Matéria: {materia} ---")
    print(f"Total de amostras (linhas): {total_amostras_materia}")
    
    # Se a matéria não tiver amostras, pula para evitar divisão por zero
    if total_amostras_materia == 0:
        continue
        
    contagem_classes = df_count_temas['tema'].value_counts()
    
    # --- Cenário 1: X >= 2 ---
    # Soma a quantidade de amostras dos temas que aparecem 2 ou mais vezes
    amostras_X2 = contagem_classes[contagem_classes >= 2].sum()
    proporcao_X2 = amostras_X2 / total_amostras_materia
    print(f"Para X >= 2:")
    print(f"  - Qtd de amostras: {amostras_X2}")
    print(f"  - Proporção sobre o total da matéria: {proporcao_X2:.4f} ({proporcao_X2:.2%})")
    
    # --- Cenário 2: X >= 5 ---
    # Soma a quantidade de amostras dos temas que aparecem 5 ou mais vezes
    amostras_X5 = contagem_classes[contagem_classes >= 5].sum()
    proporcao_X5 = amostras_X5 / total_amostras_materia
    print(f"Para X >= 5:")
    print(f"  - Qtd de amostras: {amostras_X5}")
    print(f"  - Proporção sobre o total da matéria: {proporcao_X5:.4f} ({proporcao_X5:.2%})")


--- Matéria: PLP_68_2024 ---
Total de amostras (linhas): 1974
Para X >= 2:
  - Qtd de amostras: 1893
  - Proporção sobre o total da matéria: 0.9590 (95.90%)
Para X >= 5:
  - Qtd de amostras: 1679
  - Proporção sobre o total da matéria: 0.8506 (85.06%)

--- Matéria: MPV_612_2013 ---
Total de amostras (linhas): 220
Para X >= 2:
  - Qtd de amostras: 179
  - Proporção sobre o total da matéria: 0.8136 (81.36%)
Para X >= 5:
  - Qtd de amostras: 157
  - Proporção sobre o total da matéria: 0.7136 (71.36%)

--- Matéria: PEC_6_2019 ---
Total de amostras (linhas): 268
Para X >= 2:
  - Qtd de amostras: 264
  - Proporção sobre o total da matéria: 0.9851 (98.51%)
Para X >= 5:
  - Qtd de amostras: 245
  - Proporção sobre o total da matéria: 0.9142 (91.42%)


## Clustering

In [7]:
import os
import gc
import numpy as np
import pandas as pd
from scipy.sparse import load_npz
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import adjusted_rand_score, v_measure_score
from sklearn.preprocessing import normalize
from tqdm import tqdm

# =========================
# CONFIG & PERSISTÊNCIA
# =========================
caminho_parquet = "../../data/datasets/resultados_clustering_25.parquet"
os.makedirs(os.path.dirname(caminho_parquet), exist_ok=True)

def extrair_tipo_texto(embedding_name: str) -> str:
    """Extrai a variante de texto do identificador do embedding/representação."""
    if "sem_justificativa" in embedding_name:
        return "texto_preprocessado_sem_justificativa"
    elif "texto_preprocessado" in embedding_name:
        return "texto_preprocessado"
    return "texto"

colunas_df_all = ["nivel", "embedding", "modelo", "ari", "v_measure", "materia", "tipo_texto"]

if os.path.exists(caminho_parquet):
    df_all = pd.read_parquet(caminho_parquet)
    print(f"Resultados existentes carregados: {len(df_all)} registros.")
else:
    df_all = pd.DataFrame(columns=colunas_df_all)
    print("Nenhum resultado prévio encontrado. Iniciando do zero.")

def ja_processado(df: pd.DataFrame, mat: str, niv: str, emb: str) -> bool:
    if df.empty:
        return False
    return ((df["materia"] == mat) & (df["nivel"] == niv) & (df["embedding"] == emb)).any()

materias = ["PEC_6_2019", "MPV_612_2013", "PLP_68_2024"]

# =========================
# BM25L
# =========================
bm25_matrices_raw = {
    "texto": load_npz("../../data/bm25l/doc_term_matrices/bm25l_texto_raw.npz"),
    "texto_preprocessado": load_npz("../../data/bm25l/doc_term_matrices/bm25l_texto_preprocessado_raw.npz"),
    "texto_preprocessado_sem_justificativa": load_npz("../../data/bm25l/doc_term_matrices/bm25l_texto_preprocessado_sem_justificativa_raw.npz"),
}

bm25_matrices_preprocess = {
    "texto": load_npz("../../data/bm25l/doc_term_matrices/bm25l_texto_preprocess.npz"),
    "texto_preprocessado": load_npz("../../data/bm25l/doc_term_matrices/bm25l_texto_preprocessado_preprocess.npz"),
    "texto_preprocessado_sem_justificativa": load_npz("../../data/bm25l/doc_term_matrices/bm25l_texto_preprocessado_sem_justificativa_preprocess.npz"),
}

sparse_representations = {}
for nome, X in bm25_matrices_raw.items():
    sparse_representations[f"bm25l_raw__{nome}"] = X

for nome, X in bm25_matrices_preprocess.items():
    sparse_representations[f"bm25l_preprocess__{nome}"] = X

# =========================
# LOOP PRINCIPAL
# =========================
for materia in materias:
    print(f"\n{'='*80}\nProcessando {materia}\n{'='*80}")

    mask_materia = (embeddings["materia"] == materia).values
    df_materia = embeddings[mask_materia].copy().reset_index(drop=True)

    niveis = ["tema"]
    if materia == "PLP_68_2024":
        niveis = ["tema_macro", "tema_nivel_2", "tema"]

    # Cache de Embeddings Densos (Normalizados)
    dense_cache = {}
    for emb_col in tqdm(embedding_cols, desc="Embeddings cache"):
        arr = np.ascontiguousarray(np.vstack(df_materia[emb_col].values), dtype=np.float32)
        dense_cache[emb_col] = normalize(arr, norm="l2", copy=False)

    # Cache de BM25 (Normalizado e Densa Float32)
    bm25_dense_cache = {}
    for nome, X_full in tqdm(sparse_representations.items(), desc="BM25 cache"):
        X_sub = normalize(X_full[mask_materia], norm="l2", copy=False)
        bm25_dense_cache[nome] = np.ascontiguousarray(X_sub.toarray(), dtype=np.float32)
        del X_sub

    for nivel in niveis:
        y_true = df_materia[nivel].fillna("").astype(str).values
        n_clusters = len(np.unique(y_true))
        print(f"\nNivel={nivel} | Clusters={n_clusters}")

        # 1. Avalia Embeddings Densos
        for emb_col in tqdm(embedding_cols, desc=f"{nivel} | Embeddings"):
            if ja_processado(df_all, materia, nivel, emb_col):
                continue

            X = dense_cache[emb_col]
            model = AgglomerativeClustering(n_clusters=n_clusters, metric="cosine", linkage="average")
            y_pred = model.fit_predict(X)

            novo_registro = pd.DataFrame([{
                "nivel": nivel,
                "embedding": emb_col,
                "modelo": "agglomerative",
                "ari": adjusted_rand_score(y_true, y_pred),
                "v_measure": v_measure_score(y_true, y_pred),
                "materia": materia,
                "tipo_texto": extrair_tipo_texto(emb_col),
            }])

            df_all = pd.concat([df_all, novo_registro], ignore_index=True)
            df_all.to_parquet(caminho_parquet, index=False)

            del y_pred, model

        # 2. Avalia BM25
        for nome, X_dense in tqdm(bm25_dense_cache.items(), desc=f"{nivel} | BM25"):
            if ja_processado(df_all, materia, nivel, nome):
                continue

            model = AgglomerativeClustering(n_clusters=n_clusters, metric="cosine", linkage="average")
            y_pred = model.fit_predict(X_dense)

            novo_registro = pd.DataFrame([{
                "nivel": nivel,
                "embedding": nome,
                "modelo": "agglomerative",
                "ari": adjusted_rand_score(y_true, y_pred),
                "v_measure": v_measure_score(y_true, y_pred),
                "materia": materia,
                "tipo_texto": extrair_tipo_texto(nome),
            }])

            df_all = pd.concat([df_all, novo_registro], ignore_index=True)
            df_all.to_parquet(caminho_parquet, index=False)

            del y_pred, model

    del dense_cache, bm25_dense_cache, df_materia
    gc.collect()

print(f"\nProcessamento concluído. Total de registros salvos: {len(df_all)}")
display(df_all.head(20))

Nenhum resultado prévio encontrado. Iniciando do zero.

Processando PEC_6_2019


BM25 cache: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6/6 [00:00<00:00, 30.46it/s]



Nivel=tema | Clusters=29


tema | Embeddings:   0%|                                                                                                                                             | 0/57 [00:00<?, ?it/s]C:\Users\3675-Robson\AppData\Local\Temp\ipykernel_9800\347289383.py:113: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_all = pd.concat([df_all, novo_registro], ignore_index=True)
tema | Embeddings: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 57/57 [00:05<00:00, 10.43it/s]
tema | BM25: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6/6 [00:24<00:00,  4.04s/it]



Processando MPV_612_2013


...

tema_nivel_2 | Embeddings: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 57/57 [03:37<00:00,  3.81s/it]
tema_nivel_2 | BM25: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6/6 [17:15<00:00, 172.59s/it]



Nivel=tema | Clusters=261


tema | Embeddings: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 57/57 [03:37<00:00,  3.81s/it]
tema | BM25: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6/6 [17:12<00:00, 172.15s/it]


Processamento concluído. Total de registros salvos: 315


,nivel,embedding,modelo,ari,v_measure,materia,tipo_texto
0,tema,embedding__codefuse_ai__F2LLM_v2_14B__texto,agglomerative,0.254173,0.584464,PEC_6_2019,texto
1,tema,embedding__codefuse_ai__F2LLM_v2_14B__texto_pr...,agglomerative,0.528830,0.786561,PEC_6_2019,texto_preprocessado
2,tema,embedding__codefuse_ai__F2LLM_v2_14B__texto_pr...,agglomerative,0.164449,0.567194,PEC_6_2019,texto_preprocessado_sem_justificativa
3,tema,embedding__Octen__Octen_Embedding_8B__texto,agglomerative,0.433797,0.768525,PEC_6_2019,texto
4,tema,embedding__Octen__Octen_Embedding_8B__texto_pr...,agglomerative,0.416230,0.748648,PEC_6_2019,texto_preprocessado
5,tema,embedding__Octen__Octen_Embedding_8B__texto_pr...,agglomerative,0.190014,0.588188,PEC_6_2019,texto_preprocessado_sem_justificativa
6,tema,embedding__Qwen__Qwen3_Embedding_8B__texto,agglomerative,0.440935,0.767465,PEC_6_2019,texto
7,tema,embedding__Qwen__Qwen3_Embedding_8B__texto_pre...,agglomerative,0.466991,0.780961,PEC_6_2019,texto_preprocessado
8,tema,embedding__Qwen__Qwen3_Embedding_8B__texto_pre...,agglomerative,0.196153,0.599494,PEC_6_2019,texto_preprocessado_sem_justificativa
9,tema,embedding__nvidia__llama_embed_nemotron_8b__texto,agglomerative,0.245558,0.619179,PEC_6_2019,texto


In [40]:

df_all[(df_all["nivel"] == "tema") & (df_all.embedding.str.contains("bm25"))]

,nivel,embedding,modelo,ari,v_measure,materia,tipo_texto,familia
57,tema,bm25l_raw__texto,agglomerative,0.236963,0.554977,PEC_6_2019,texto,bm25
58,tema,bm25l_raw__texto_preprocessado,agglomerative,0.265479,0.605082,PEC_6_2019,texto_preprocessado,bm25
59,tema,bm25l_raw__texto_preprocessado_sem_justificativa,agglomerative,0.098848,0.438801,PEC_6_2019,texto_preprocessado_sem_justificativa,bm25
60,tema,bm25l_preprocess__texto,agglomerative,0.223270,0.570546,PEC_6_2019,texto,bm25
61,tema,bm25l_preprocess__texto_preprocessado,agglomerative,0.270408,0.609462,PEC_6_2019,texto_preprocessado,bm25
62,tema,bm25l_preprocess__texto_preprocessado_sem_just...,agglomerative,0.149266,0.500277,PEC_6_2019,texto_preprocessado_sem_justificativa,bm25
120,tema,bm25l_raw__texto,agglomerative,0.087438,0.607368,MPV_612_2013,texto,bm25
121,tema,bm25l_raw__texto_preprocessado,agglomerative,0.185694,0.688855,MPV_612_2013,texto_preprocessado,bm25
122,tema,bm25l_raw__texto_preprocessado_sem_justificativa,agglomerative,0.145400,0.673976,MPV_612_2013,texto_preprocessado_sem_justificativa,bm25
123,tema,bm25l_preprocess__texto,agglomerative,0.108312,0.623016,MPV_612_2013,texto,bm25


In [41]:
import pandas as pd
import numpy as np

# 1. Filtrar a base
df_filtered = df_all[
    (df_all["nivel"] == "tema") & 
    (df_all["tipo_texto"] == "texto_preprocessado")
].copy()

# 2. Mapeamento dos nomes brutos para os nomes exatos do LaTeX
def format_model_name(name):
    if "bm25l" in name:
        if "raw" in name.split("__")[0]:
            return "BM25L"
        return "BM25L (w/ pre-proces.)"
    if "F2LLM_v2_14B" in name:
        return "F2LLM-v2-14B"
    if "harrier_oss_v1_27b" in name:
        return "harrier-oss-v1-27b"
    if "ICT_TIME_and_Querit" in name:
        return "ICT-TIME-and-Querit"
    if "jina_embeddings_v5_text_small" in name:
        if "clustering" in name:
            return "jina-v5-text-small (cls)"
        return "jina-v5-text-small (tm)"
    if "KaLM_Embedding_Gemma3_12B" in name:
        return "KaLM-Gemma3-12B"
    if "llama_embed_nemotron_8b" in name:
        return "nemotron-8B"
    if "Octen_Embedding_8B" in name:
        return "Octen-Embedding-8B"
    if "Qwen3_Embedding_8B" in name:
        return "Qwen3-Embedding-8B"
    if "gemini_embedding_2" in name:
        if "sts" in name or "similarity" in name:
            return "gemini-embedding-2 (ss)"
        if "clustering" in name or "cls" in name:
            return "gemini-embedding-2 (cls)"
        return "gemini-embedding-2 (none)"
    if "text_embedding_3_large" in name:
        return "text-embedding-3-large"
    if "bertimbau_tuned" in name:
        if "mean" in name or "mp" in name:
            return "BERTimbau-FT (mp)"
        return "BERTimbau-FT (trunc)"
    if "serafim" in name:
        is_ir = "ir" in name.lower()
        strategy = "mp" if ("mean" in name or "mp" in name) else "trunc"
        prefix = "Serafim IR" if is_ir else "Serafim Base"
        return f"{prefix} ({strategy})"
    return name

df_filtered["Model"] = df_filtered["embedding"].apply(format_model_name)

# 3. Criar a Tabela Pivotada
# Métricas a extrair
metrics = ["ari", "v_measure"]

df_pivot = df_filtered.pivot_table(
    index="Model",
    columns="materia",
    values=metrics,
    aggfunc="first"
)

# 4. Estruturar a ordem exata das colunas do LaTeX
# (MPV: P@1, R-P, MAP) | (PEC: P@1, R-P, MAP) | (PLP: P@1, R-P, MAP, NDGC@all)
cols_order = [
    ("ari", "MPV_612_2013"),
    ("v_measure", "MPV_612_2013"),
    ("ari", "PEC_6_2019"),
    ("v_measure", "PEC_6_2019"),
    ("ari", "PLP_68_2024"),
    ("v_measure", "PLP_68_2024"),
]

# Garantir que todas as colunas existem (preenche NaN se faltar)
for col in cols_order:
    if col not in df_pivot.columns:
        df_pivot[col] = np.nan

df_pivot = df_pivot[cols_order]

# Renomear colunas para cabeçalho duplo limpo
df_pivot.columns = pd.MultiIndex.from_tuples([
    ("MPV", "ARI"), ("MPV", "V-Meas."),
    ("PEC", "ARI"), ("PEC", "V-Meas."),
    ("PLP", "ARI"), ("PLP", "V-Meas."),
])

# 5. Definir a ordem exata das linhas conforme o artigo
model_order = [
    # Lexical Baselines
    "BM25L",
    "BM25L (w/ pre-proces.)",
    # Open-Weight Multilingual
    "F2LLM-v2-14B",
    "harrier-oss-v1-27b",
    "ICT-TIME-and-Querit",
    "jina-v5-text-small (tm)",
    "jina-v5-text-small (cls)",
    "KaLM-Gemma3-12B",
    "nemotron-8B",
    "Octen-Embedding-8B",
    "Qwen3-Embedding-8B",
    # Proprietary APIs
    "gemini-embedding-2 (none)",
    "gemini-embedding-2 (ss)",
    "gemini-embedding-2 (cls)",
    "text-embedding-3-large",
    # Domain / Specialized
    "BERTimbau-FT (mp)",
    "BERTimbau-FT (trunc)",
    "Serafim Base (mp)",
    "Serafim Base (trunc)",
    "Serafim IR (mp)",
    "Serafim IR (trunc)"
]

# Reindexar preservando apenas os que existem
existing_models = [m for m in model_order if m in df_pivot.index]
df_pivot = df_pivot.reindex(existing_models)

# 6. Exibição Formatada no Notebook (3 casas decimais)
display(df_pivot.style.format("{:.3f}", na_rep="--"))

In [57]:
df_exp = df_all[
    (df_all["tipo_texto"] == "texto_preprocessado") & 
    (df_all["nivel"] == "tema")
].copy()

# Lista de métricas a ranquear (retrieval e clustering)
metricas = ["ari", "v_measure"]

# Filtra apenas as métricas presentes nas colunas
metricas_validas = [m for m in metricas if m in df_exp.columns]

# =====================================================================
# 2. TRANSFORMAÇÃO PARA FORMATO LONGO E CÁLCULO DOS RANKS
# =====================================================================
# Derrete o DataFrame para termos 1 linha por (materia, embedding, metrica)
df_melted = df_exp.melt(
    id_vars=["materia", "embedding"],
    value_vars=metricas_validas,
    var_name="metrica",
    value_name="score"
).dropna(subset=["score"])

# Calcula a posição (Rank 1 = Maior Score) dentro de cada grupo (materia, metrica)
# method='min' garante ranking padrão de competição em caso de empate
df_melted["rank"] = (
    df_melted.groupby(["materia", "metrica"])["score"]
    .rank(ascending=False, method="min")
    .astype(int)
)

# =====================================================================
# 3. TABELA DE FREQUÊNCIA DE POSIÇÕES (TOP 1, TOP 2, TOP 3, ...)
# =====================================================================
# Cria a matriz: Linha = Modelo/Embedding | Coluna = Posição no Rank
df_ranks_count = pd.crosstab(
    index=df_melted["embedding"],
    columns=df_melted["rank"],
    margins=False
)

# Renomeia as colunas para formato amigável (Top 1, Top 2, ...)
df_ranks_count.columns = [f"Top {col}" for col in df_ranks_count.columns]

# Adiciona Rank Médio para ordenação global consistente
rank_medio = df_melted.groupby("embedding")["rank"].mean().rename("Rank Médio")
df_ranks_count["Rank Médio"] = rank_medio.round(2)

# Ordena a tabela pelos mais frequentes no Top 1 e depois pelo melhor Rank Médio
df_ranks_count = df_ranks_count.sort_values(by=[ "Rank Médio", "Top 1"], ascending=[True, False])

# Preenche posições vazias com 0
df_ranks_count = df_ranks_count.fillna(0).astype({col: int for col in df_ranks_count.columns if col != "Rank Médio"})

# =====================================================================
# 4. EXIBIÇÃO DOS RESULTADOS
# =====================================================================
print("=" * 80)
print("DISTRIBUIÇÃO DE POSIÇÕES POR MODELO (Todas as Matérias e Métricas)")
print("=" * 80)
display(df_ranks_count)

DISTRIBUIÇÃO DE POSIÇÕES POR MODELO (Todas as Matérias e Métricas)


,Top 1,Top 2,Top 3,Top 4,Top 5,Top 6,Top 7,Top 8,Top 9,Top 10,...,Top 13,Top 14,Top 15,Top 16,Top 17,Top 18,Top 19,Top 20,Top 21,Rank Médio
embedding,,,,,,,,,,,,,,,,,,,,,
embedding__joaorobson__harrier_oss_v1_27b__texto_preprocessado,1,1,1,3,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,3.00
embedding__gemini__gemini_embedding_2__sts__texto_preprocessado,1,3,0,1,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,3.50
embedding__Octen__Octen_Embedding_8B__texto_preprocessado,1,1,0,1,3,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,3.67
embedding__codefuse_ai__F2LLM_v2_14B__texto_preprocessado,2,0,1,1,0,0,1,0,1,0,...,0,0,0,0,0,0,0,0,0,4.17
embedding__ICT_TIME_and_Querit__ICT_TIME_and_Querit_embedding_v1__texto_preprocessado,1,1,0,0,1,3,0,0,0,0,...,0,0,0,0,0,0,0,0,0,4.33
embedding__Qwen__Qwen3_Embedding_8B__texto_preprocessado,0,0,2,0,0,1,0,1,0,0,...,0,0,0,0,0,0,0,0,0,7.17
embedding__openai__text_embedding_3_large__texto_preprocessado,0,0,0,0,0,1,3,1,0,0,...,0,0,0,0,0,0,0,0,0,7.67
embedding__nvidia__llama_embed_nemotron_8b__texto_preprocessado,0,0,0,0,0,0,1,3,2,0,...,0,0,0,0,0,0,0,0,0,8.17
bm25l_preprocess__texto_preprocessado,0,0,1,0,2,0,0,0,0,1,...,1,0,1,0,0,0,0,0,0,8.50


## Avaliação da segmentação do texto

In [42]:
MODEL_SIZE = {
    "F2LLM_v2_14B": 14.0,
    "Octen_Embedding_8B": 8.0,
    "Qwen3_Embedding_8B": 8.0,
    "llama_embed_nemotron_8b": 8.0,
    "harrier_oss_v1_27b": 27.0,
    "KaLM_Embedding_Gemma3_12B_2511": 12.0,
    "jina_embeddings_v5_text_small": 0.6,      # 330M
    "serafim_335m_portuguese_pt_sentence_encoder": 0.335,
    "serafim_335m_portuguese_pt_sentence_encoder_ir": 0.335,
    "bertimbau_tuned": 0.335,                   # BERTimbau Large ≈335M
    "ICT_TIME_and_Querit_embedding_v1": 4
}


def get_model_size(name):
    for model, size in MODEL_SIZE.items():
        if model in name:
            return size
    return np.nan

MODEL_FAMILY = {
    "F2LLM_v2_14B": "multilingual",
    "Octen_Embedding_8B": "multilingual",
    "Qwen3_Embedding_8B": "multilingual",
    "llama_embed_nemotron_8b": "multilingual",
    "harrier_oss_v1_27b": "multilingual",
    "KaLM_Embedding_Gemma3_12B_2511": "multilingual",
    "jina_embeddings_v5_text_small": "multilingual",      # 330M
    "serafim_335m_portuguese_pt_sentence_encoder": "specialized",
    "serafim_335m_portuguese_pt_sentence_encoder_ir": "specialized",
    "bertimbau_tuned": "specialized",                   # BERTimbau Large ≈335M
    "ICT_TIME_and_Querit_embedding_v1": "multilingual",
    "bm25l": "bm25",
    "gemini": "proprietary",
    "openai": "proprietary"
}

def get_model_family(name):
    for model, size in MODEL_FAMILY.items():
        if model in name:
            return size
    return np.nan

In [24]:
df_all["familia"] = df_all.embedding.apply(get_model_family)

In [16]:
metricas = ["ari", "v_measure"]


In [43]:
(
    df_all[df_all["nivel"] == "tema"]
        .groupby(["materia", "tipo_texto","familia"])[metricas]
    .agg(["mean", "std"])
)

ari  \
                                                                     mean   
materia      tipo_texto                            familia                  
MPV_612_2013 texto                                 bm25          0.097875   
                                                   multilingual  0.174945   
                                                   proprietary   0.163295   
                                                   specialized   0.093655   
             texto_preprocessado                   bm25          0.204867   
                                                   multilingual  0.203190   
                                                   proprietary   0.179645   
                                                   specialized   0.119103   
             texto_preprocessado_sem_justificativa bm25          0.147760   
                                                   multilingual  0.145785   
                                                   proprietary   0.222292   
                                                   specialized   0.085637   
PEC_6_2019   texto                                 bm25          0.230117   
                                                   multilingual  0.313697   
                                                   proprietary   0.216735   
                                                   specialized   0.169400   
             texto_preprocessado                   bm25          0.267943   
                                                   multilingual  0.367954   
                                                   proprietary   0.343770   
                                                   specialized   0.201861   
             texto_preprocessado_sem_justificativa bm25          0.124057   
                                                   multilingual  0.153939   
                                                   proprietary   0.148674   
                                                   specialized   0.091768   
PLP_68_2024  texto                                 bm25          0.352453   
                                                   multilingual  0.323107   
                                                   proprietary   0.265892   
                                                   specialized   0.131576   
             texto_preprocessado                   bm25          0.355745   
                                                   multilingual  0.347922   
                                                   proprietary   0.338743   
                                                   specialized   0.133121   
             texto_preprocessado_sem_justificativa bm25          0.337957   
                                                   multilingual  0.241286   
                                                   proprietary   0.299884   
                                                   specialized   0.045485   

                                                                           \
                                                                      std   
materia      tipo_texto                            familia                  
MPV_612_2013 texto                                 bm25          0.014760   
                                                   multilingual  0.046266   
                                                   proprietary   0.044258   
                                                   specialized   0.040348   
             texto_preprocessado                   bm25          0.027115   
                                                   multilingual  0.062648   
                                                   proprietary   0.031554   
                                                   specialized   0.014988   
             texto_preprocessado_sem_justificativa bm25          0.003338   
                                                   multilingual  0.031876   
                                                   proprietary   

In [44]:
import pandas as pd
import numpy as np

# 1. Filtra o nível 'tema' e aplica o mapeamento de família
df_analise = df_all[df_all["nivel"] == "tema"].copy()
df_analise["familia"] = df_analise["embedding"].apply(get_model_family)

# 2. Calcula as médias por matéria, família e tipo de texto
metricas = ["ari", "v_measure"]
df_medias = (
    df_analise
    .groupby(["materia", "familia", "tipo_texto"])[metricas]
    .mean()
    .reset_index()
)

# 3. Separa o Baseline (Texto Bruto) e os Tratamentos
df_bruto = df_medias[df_medias["tipo_texto"] == "texto"].copy()
df_alt_just = df_medias[df_medias["tipo_texto"] == "texto_preprocessado"].copy()
df_sem_just = df_medias[df_medias["tipo_texto"] == "texto_preprocessado_sem_justificativa"].copy()

# 4. Faz o merge pareado por Matéria e Família
m_just = pd.merge(df_alt_just, df_bruto, on=["materia", "familia"], suffixes=("_alt_just", "_bruto"))
m_sem = pd.merge(df_sem_just, df_bruto, on=["materia", "familia"], suffixes=("_sem_just", "_bruto"))

# 5. Constrói o relatório detalhado de ganhos (Deltas)
registros = []

for materia in df_analise["materia"].unique():
    for fam in df_analise["familia"].dropna().unique():
        sub_j = m_just[(m_just["materia"] == materia) & (m_just["familia"] == fam)]
        sub_s = m_sem[(m_sem["materia"] == materia) & (m_sem["familia"] == fam)]
        
        if sub_j.empty or sub_s.empty:
            continue
            
        for metrica in metricas:
            b = sub_j[f"{metrica}_bruto"].values[0]
            j = sub_j[f"{metrica}_alt_just"].values[0]
            s = sub_s[f"{metrica}_sem_just"].values[0]
            
            delta_just = j - b
            pct_just = (delta_just / b) * 100 if b != 0 else 0
            
            delta_sem = s - b
            pct_sem = (delta_sem / b) * 100 if b != 0 else 0
            
            registros.append({
                "Matéria": materia,
                "Família": fam,
                "Métrica": metrica,
                "Texto Bruto": round(b, 4),
                "Alt + Just": round(j, 4),
                "Δ (Alt+Just)": round(delta_just, 4),
                "% Ganho (Alt+Just)": f"{pct_just:+.2f}%",
                "Alt Pura": round(s, 4),
                "Δ (Alt Pura)": round(delta_sem, 4),
                "% Ganho (Alt Pura)": f"{pct_sem:+.2f}%"
            })

df_ganhos = pd.DataFrame(registros)

# Ordena para facilitar a inspeção
df_ganhos = df_ganhos.sort_values(by=["Matéria", "Família", "Métrica"])

# Exibe o resultado completo
display(df_ganhos)

,Matéria,Família,Métrica,Texto Bruto,Alt + Just,Δ (Alt+Just),% Ganho (Alt+Just),Alt Pura,Δ (Alt Pura),% Ganho (Alt Pura)
14,MPV_612_2013,bm25,ari,0.0979,0.2049,0.1070,+109.31%,0.1478,0.0499,+50.97%
15,MPV_612_2013,bm25,v_measure,0.6152,0.7004,0.0852,+13.85%,0.6765,0.0613,+9.97%
8,MPV_612_2013,multilingual,ari,0.1749,0.2032,0.0282,+16.15%,0.1458,-0.0292,-16.67%
9,MPV_612_2013,multilingual,v_measure,0.6563,0.6876,0.0313,+4.77%,0.6613,0.0051,+0.77%
10,MPV_612_2013,proprietary,ari,0.1633,0.1796,0.0164,+10.01%,0.2223,0.0590,+36.13%
11,MPV_612_2013,proprietary,v_measure,0.6398,0.6836,0.0438,+6.85%,0.7086,0.0688,+10.75%
12,MPV_612_2013,specialized,ari,0.0937,0.1191,0.0254,+27.17%,0.0856,-0.0080,-8.56%
13,MPV_612_2013,specialized,v_measure,0.5667,0.6214,0.0547,+9.65%,0.6017,0.0350,+6.17%
6,PEC_6_2019,bm25,ari,0.2301,0.2679,0.0378,+16.44%,0.1241,-0.1061,-46.09%
7,PEC_6_2019,bm25,v_measure,0.5628,0.6073,0.0445,+7.91%,0.4695,-0.0932,-16.57%


In [45]:
# 1. Garante a conversão da coluna de % para float numérico
df_ganhos["pct_num_Alt_Just"] = (
    df_ganhos["% Ganho (Alt+Just)"]
    .str.rstrip("%")
    .astype(float)
)
df_ganhos["pct_num_Alt_SemJust"] = (
    df_ganhos["% Ganho (Alt Pura)"]
    .str.rstrip("%")
    .astype(float)
)

# 2. Agrupa por Família e Métrica agregando médias
df_ganhos_familia = (
    df_ganhos
    .groupby(["Família", "Métrica"])[
        ["Texto Bruto", "Alt + Just", "Δ (Alt+Just)", "pct_num_Alt_Just", 
         "Alt Pura", "Δ (Alt Pura)", "pct_num_Alt_SemJust"]
    ]
    .mean()
    .round(4)
    .reset_index()
)

# 3. Formata as colunas percentuais de volta para string com sinal
df_ganhos_familia["% Médio (Alt+Just)"] = df_ganhos_familia["pct_num_Alt_Just"].apply(lambda x: f"{x:+.2f}%")
df_ganhos_familia["% Médio (Alt Pura)"] = df_ganhos_familia["pct_num_Alt_SemJust"].apply(lambda x: f"{x:+.2f}%")

# 4. Seleciona e organiza as colunas finais
cols_exibicao = [
    "Família", "Métrica", 
    "Texto Bruto", "Alt + Just", "Δ (Alt+Just)", "% Médio (Alt+Just)",
    "Alt Pura", "Δ (Alt Pura)", "% Médio (Alt Pura)"
]
df_ganhos_familia = df_ganhos_familia[cols_exibicao].sort_values(by=["Família", "Métrica"])

display(df_ganhos_familia)

,Família,Métrica,Texto Bruto,Alt + Just,Δ (Alt+Just),% Médio (Alt+Just),Alt Pura,Δ (Alt Pura),% Médio (Alt Pura)
0,bm25,ari,0.2268,0.2762,0.0494,+42.23%,0.2033,-0.0236,+0.26%
1,bm25,v_measure,0.6622,0.7058,0.0437,+7.31%,0.6532,-0.0089,-1.99%
2,multilingual,ari,0.2706,0.3064,0.0358,+13.71%,0.1803,-0.0903,-30.97%
3,multilingual,v_measure,0.7044,0.7347,0.0302,+4.51%,0.6574,-0.0470,-6.85%
4,proprietary,ari,0.2153,0.2874,0.0721,+32.01%,0.2236,0.0083,+5.84%
5,proprietary,v_measure,0.6558,0.7259,0.0701,+11.52%,0.6829,0.0271,+3.88%
6,specialized,ari,0.1316,0.1514,0.0198,+15.83%,0.0743,-0.0572,-39.94%
7,specialized,v_measure,0.5843,0.6263,0.0421,+7.67%,0.5507,-0.0336,-5.65%


In [46]:
# 1. Garante a conversão das colunas percentuais para float
df_ganhos["pct_num_Alt_Just"] = (
    df_ganhos["% Ganho (Alt+Just)"]
    .str.rstrip("%")
    .astype(float)
)
df_ganhos["pct_num_Alt_SemJust"] = (
    df_ganhos["% Ganho (Alt Pura)"]
    .str.rstrip("%")
    .astype(float)
)

# 2. Agrupa apenas por Métrica agregando médias e desvios
colunas_num = [
    "Texto Bruto", "Alt + Just", "Δ (Alt+Just)", "pct_num_Alt_Just",
    "Alt Pura", "Δ (Alt Pura)", "pct_num_Alt_SemJust"
]

df_resumo_metrica = (
    df_ganhos
    .groupby("Métrica")[colunas_num]
    .agg(["mean", "std"])
    .round(4)
)

# 3. Cria o DataFrame consolidado ordenado pelo maior ganho
df_ganhos_por_metrica = pd.DataFrame(index=df_resumo_metrica.index)

df_ganhos_por_metrica["Texto Bruto"] = df_resumo_metrica[("Texto Bruto", "mean")]
df_ganhos_por_metrica["Alt + Just"] = df_resumo_metrica[("Alt + Just", "mean")]
df_ganhos_por_metrica["Δ Médio (Alt+Just)"] = df_resumo_metrica[("Δ (Alt+Just)", "mean")]
df_ganhos_por_metrica["% Ganho Médio (Alt+Just)"] = df_resumo_metrica[("pct_num_Alt_Just", "mean")]
df_ganhos_por_metrica["Alt Pura"] = df_resumo_metrica[("Alt Pura", "mean")]
df_ganhos_por_metrica["Δ Médio (Alt Pura)"] = df_resumo_metrica[("Δ (Alt Pura)", "mean")]
df_ganhos_por_metrica["% Ganho Médio (Alt Pura)"] = df_resumo_metrica[("pct_num_Alt_SemJust", "mean")]

# Ordena pelo maior ganho percentual obtido com Alt + Just
df_ganhos_por_metrica = df_ganhos_por_metrica.sort_values(
    by="% Ganho Médio (Alt+Just)", 
    ascending=False
).reset_index()

# 4. Formata as colunas percentuais para exibição
df_ganhos_por_metrica["% Ganho Médio (Alt+Just)"] = df_ganhos_por_metrica["% Ganho Médio (Alt+Just)"].apply(lambda x: f"{x:+.2f}%")
df_ganhos_por_metrica["% Ganho Médio (Alt Pura)"] = df_ganhos_por_metrica["% Ganho Médio (Alt Pura)"].apply(lambda x: f"{x:+.2f}%")

display(df_ganhos_por_metrica)

,Métrica,Texto Bruto,Alt + Just,Δ Médio (Alt+Just),% Ganho Médio (Alt+Just),Alt Pura,Δ Médio (Alt Pura),% Ganho Médio (Alt Pura)
0,ari,0.2111,0.2553,0.0443,+25.94%,0.1704,-0.0407,-16.20%
1,v_measure,0.6517,0.6982,0.0465,+7.75%,0.6360,-0.0156,-2.65%


In [50]:
import gc
import scipy.stats as stats
import pandas as pd
import numpy as np

# =====================================================================
# 1. CONSOLIDAÇÃO DAS ESTATÍSTICAS DESCRITIVAS (MÉDIA E DESVIO PADRÃO)
# =====================================================================
print("--- Calculando Média e Desvio Padrão por Tratamento ---")

print("\n" + "="*70)
print("INICIANDO ANÁLISE ESTATÍSTICA COMPLETA")
print("="*70)
metricas = ["ari", "v_measure"]
# =====================================================================
# 2. CONSOLIDAÇÃO DAS ESTATÍSTICAS DESCRITIVAS (MÉDIA E DESVIO PADRÃO)
# =====================================================================
print("\n--- 1. Calculando Média e Desvio Padrão por Tratamento ---")

df_estatisticas = (
    df_all[df_all["nivel"] == "tema"]
    .groupby(["materia", "tipo_texto"])[metricas]
    .agg(["mean", "std"])
)

# Simplifica o multi-index das colunas para 'metrica_mean' e 'metrica_std'
df_estatisticas.columns = [f"{metric}_{stat}" for metric, stat in df_estatisticas.columns]
display(df_estatisticas)


# =====================================================================
# 3. TESTES DE SIGNIFICÂNCIA ESTATÍSTICA (MELHORA EM RELAÇÃO AO BRUTO)
# =====================================================================
print("\n--- 2. Executando Testes de Wilcoxon (Melhora vs. Bruto) ---")

# Filtra apenas o nível 'tema'
df_filtrado = df_all[df_all["nivel"] == "tema"].copy()
lista_materias = df_filtrado["materia"].unique()
registros_testes = []

for materia in lista_materias:
    df_mat = df_filtrado[df_filtrado["materia"] == materia]
    
    # Isola o baseline fixo: Texto Bruto
    df_bruto = df_mat[df_mat["tipo_texto"] == "texto"]
    
    if df_bruto.empty:
        print(f"[Aviso] Matéria {materia} não possui dados para o tipo 'texto' (Bruto).")
        continue
        
    for metric in metricas:
        scores_bruto = df_bruto.sort_values("embedding")[metric].values
        
        # -----------------------------------------------------------------
        # Teste 1: Pré-processado COMPLETO (Alt. + Just.) SUPEROU o Bruto?
        # -----------------------------------------------------------------
        df_alt_just = df_mat[df_mat["tipo_texto"] == "texto_preprocessado"]
        if not df_alt_just.empty:
            scores_alt_just = df_alt_just.sort_values("embedding")[metric].values
            
            if len(scores_alt_just) == len(scores_bruto):
                # 'greater' testa se: scores_alt_just > scores_bruto
                _, p_val_alt_just = stats.wilcoxon(scores_alt_just, scores_bruto, alternative="greater")
            else:
                p_val_alt_just = np.nan
        else:
            p_val_alt_just = np.nan
            
        # -----------------------------------------------------------------
        # Teste 2: Pré-processado SEM Justificativa (Alt. Pura) SUPEROU o Bruto?
        # -----------------------------------------------------------------
        df_sem_j = df_mat[df_mat["tipo_texto"] == "texto_preprocessado_sem_justificativa"]
        if not df_sem_j.empty:
            scores_sem_j = df_sem_j.sort_values("embedding")[metric].values
            
            if len(scores_sem_j) == len(scores_bruto):
                # 'greater' testa se: scores_sem_j > scores_bruto
                _, p_val_sem_j = stats.wilcoxon(scores_sem_j, scores_bruto, alternative="greater")
            else:
                p_val_sem_j = np.nan
        else:
            p_val_sem_j = np.nan
            
        # Guarda os p-values calculados para esta métrica nesta matéria
        registros_testes.append({
            "materia": materia,
            "metrica": metric,
            "p_val_Alt_Just_vs_Bruto": p_val_alt_just,
            "p_val_Alt_SemJust_vs_Bruto": p_val_sem_j
        })

# Consolida os resultados em DataFrame
df_p_values = pd.DataFrame(registros_testes)

# Define o nível de significância rigoroso do seu paper (Alpha = 0.05)
ALPHA_CORTE = 0.05

# Cria as colunas de veredito se houve ganho ou não em relação ao bruto
df_p_values["Melhorou_Alt_Just?"] = df_p_values["p_val_Alt_Just_vs_Bruto"].apply(
    lambda p: "SIM (*)" if p < ALPHA_CORTE else "Não"
)
df_p_values["Melhorou_Alt_SemJust?"] = df_p_values["p_val_Alt_SemJust_vs_Bruto"].apply(
    lambda p: "SIM (*)" if p < ALPHA_CORTE else "Não"
)

# Reorganiza as colunas para melhor leitura na tela
colunas_ordenadas = [
    "materia", "metrica", 
    "p_val_Alt_Just_vs_Bruto", "Melhorou_Alt_Just?",
    "p_val_Alt_SemJust_vs_Bruto", "Melhorou_Alt_SemJust?"
]
df_p_values = df_p_values[colunas_ordenadas]

print("\n>>> TABELA DE VEREDITO FINAL (Confiança de 95%) <<<")
display(df_p_values)

# Limpeza de memória final
#del df_filtrado, registros_testes
gc.collect()

--- Calculando Média e Desvio Padrão por Tratamento ---

INICIANDO ANÁLISE ESTATÍSTICA COMPLETA

--- 1. Calculando Média e Desvio Padrão por Tratamento ---


ari_mean   ari_std  \
materia      tipo_texto                                                  
MPV_612_2013 texto                                  0.142160  0.055162   
             texto_preprocessado                    0.174840  0.056574   
             texto_preprocessado_sem_justificativa  0.143361  0.054836   
PEC_6_2019   texto                                  0.246041  0.119991   
             texto_preprocessado                    0.306368  0.124623   
             texto_preprocessado_sem_justificativa  0.132327  0.037244   
PLP_68_2024  texto                                  0.260281  0.113228   
             texto_preprocessado                    0.285547  0.112539   
             texto_preprocessado_sem_justificativa  0.205711  0.119712   

                                                    v_measure_mean  \
materia      tipo_texto                                              
MPV_612_2013 texto                                        0.623613   
             texto_preprocessado                          0.669133   
             texto_preprocessado_sem_justificativa        0.654730   
PEC_6_2019   texto                                        0.588119   
             texto_preprocessado                          0.654023   
             texto_preprocessado_sem_justificativa        0.511076   
PLP_68_2024  texto                                        0.758639   
             texto_preprocessado                          0.774720   
             texto_preprocessado_sem_justificativa        0.728223   

                                                    v_measure_std  
materia      tipo_texto                                            
MPV_612_2013 texto                                       0.057218  
             texto_preprocessado                         0.041950  
             texto_preprocessado_sem_justificativa       0.045376  
PEC_6_2019   texto                                       0.111229  
             texto_preprocessado                         0.092653  
             texto_preprocessado_sem_justificativa       0.056961  
PLP_68_2024  texto                                       0.082453  
             texto_preprocessado                         0.077668  
             texto_preprocessado_sem_justificativa       0.092764


--- 2. Executando Testes de Wilcoxon (Melhora vs. Bruto) ---

>>> TABELA DE VEREDITO FINAL (Confiança de 95%) <<<


,materia,metrica,p_val_Alt_Just_vs_Bruto,Melhorou_Alt_Just?,p_val_Alt_SemJust_vs_Bruto,Melhorou_Alt_SemJust?
0,PEC_6_2019,ari,1.241207e-03,SIM (*),0.999984,Não
1,PEC_6_2019,v_measure,9.870529e-05,SIM (*),0.999200,Não
2,MPV_612_2013,ari,7.890224e-03,SIM (*),0.580903,Não
3,MPV_612_2013,v_measure,9.536743e-07,SIM (*),0.011927,SIM (*)
4,PLP_68_2024,ari,2.142429e-03,SIM (*),0.994327,Não
5,PLP_68_2024,v_measure,1.463890e-04,SIM (*),0.972653,Não


320

## Avaliação dos níves temáticos

In [53]:
# =====================================================================
# CONFIGURAÇÃO
# =====================================================================
metricas = ["ari", "v_measure"]

# Mapeamento dos níveis internos do DataFrame para os nomes do artigo
map_niveis = {
    "tema": "Original Theme",
    "tema_nivel_2": "Intermediate Theme",
    "tema_macro": "General Theme"
}
ordem_niveis = ["Original Theme", "Intermediate Theme", "General Theme"]

# Filtra apenas a matéria PLP_68_2024 na versão padrão (Changes + Justification)
df_plp = df_all[
    (df_all["materia"] == "PLP_68_2024") & 
    (df_all["tipo_texto"] == "texto_preprocessado")
].copy()

df_plp["Level"] = df_plp["nivel"].map(map_niveis)

# =====================================================================
# 1. ESTATÍSTICAS DESCRITIVAS (MEAN +- STD)
# =====================================================================
print("=" * 70)
print("ESTATÍSTICAS DESCRITIVAS POR NÍVEL TEMÁTICO (PLP 68/2024)")
print("=" * 70)

df_estatisticas_plp = (
    df_plp.groupby("Level")[metricas]
    .agg(["mean", "std"])
    .reindex(ordem_niveis)
)

display(df_estatisticas_plp)

# =====================================================================
# 2. TESTES DE WILCOXON (INTERMEDIATE E GENERAL VS. ORIGINAL THEME)
# =====================================================================
print("\n" + "=" * 70)
print("TESTES DE WILCOXON PAREADOS (vs. Original Theme)")
print("=" * 70)

# Isola o baseline (Original Theme)
df_orig = df_plp[df_plp["Level"] == "Original Theme"]
niveis_comparacao = ["Intermediate Theme", "General Theme"]
registros_testes_niveis = []

for nivel in niveis_comparacao:
    df_nivel = df_plp[df_plp["Level"] == nivel]
    
    for metric in metricas:
        # Garante o alinhamento exato dos modelos pareados
        scores_orig = df_orig.sort_values("embedding")[metric].values
        scores_nivel = df_nivel.sort_values("embedding")[metric].values
        
        if len(scores_orig) == len(scores_nivel) and len(scores_orig) > 0:
            # Teste unilateral: verifica ganho em relação ao tema original
            stat, p_val_greater = stats.wilcoxon(scores_nivel, scores_orig, alternative="greater")
            # Teste bilateral/unilateral geral para checar significância
            _, p_val_less = stats.wilcoxon(scores_nivel, scores_orig, alternative="less")
        else:
            p_val_greater = np.nan
            p_val_less = np.nan
            
        registros_testes_niveis.append({
            "Level": nivel,
            "Metric": metric,
            "Mean_Orig": np.mean(scores_orig),
            "Mean_Level": np.mean(scores_nivel),
            "p_val_greater": p_val_greater,
            "p_val_less": p_val_less,
            "p_val_min": min(p_val_greater, p_val_less) if pd.notnull(p_val_greater) else np.nan
        })

df_wilcoxon_niveis = pd.DataFrame(registros_testes_niveis)
df_wilcoxon_niveis["Significante (p < 0.05)?"] = df_wilcoxon_niveis["p_val_greater"].apply(
    lambda p: "SIM (*)" if p < 0.05 else "Não"
)

display(df_wilcoxon_niveis)

# =====================================================================
# 3. EXIBIÇÃO NO FORMATO IDÊNTICO À TABELA LATEX
# =====================================================================
print("\n" + "=" * 70)
print("LINHAS FORMATADAS PARA A TABELA LATEX")
print("=" * 70)

for nivel in ordem_niveis:
    row_str = f"{nivel} & "
    vals = []
    for m in metricas:
        mean_val = df_estatisticas_plp.loc[nivel, (m, "mean")]
        std_val = df_estatisticas_plp.loc[nivel, (m, "std")]
        
        # Identifica se houve significância estatística de aumento (apenas P@1)
        sig = ""
        if nivel != "Original Theme":
            p = df_wilcoxon_niveis[
                (df_wilcoxon_niveis["Level"] == nivel) & 
                (df_wilcoxon_niveis["Metric"] == m)
            ]["p_val_greater"].values[0]
            if p < 0.05:
                sig = "^{*}"
        
        vals.append(f"${mean_val:.3f} \\pm {std_val:.3f}{sig}$")
        
    print(row_str + " &\n".join(vals) + r" \\" + "\n")

gc.collect()

ESTATÍSTICAS DESCRITIVAS POR NÍVEL TEMÁTICO (PLP 68/2024)


ari           v_measure          
                        mean       std      mean       std
Level                                                     
Original Theme      0.285547  0.112539  0.774720  0.077668
Intermediate Theme  0.278921  0.139520  0.663187  0.096807
General Theme       0.141341  0.062008  0.390735  0.078379


TESTES DE WILCOXON PAREADOS (vs. Original Theme)


,Level,Metric,Mean_Orig,Mean_Level,p_val_greater,p_val_less,p_val_min,Significante (p < 0.05)?
0,Intermediate Theme,ari,0.285547,0.278921,0.633265,3.796439e-01,3.796439e-01,Não
1,Intermediate Theme,v_measure,0.774720,0.663187,1.000000,4.768372e-07,4.768372e-07,Não
2,General Theme,ari,0.285547,0.141341,0.999999,2.384186e-06,2.384186e-06,Não
3,General Theme,v_measure,0.774720,0.390735,1.000000,4.768372e-07,4.768372e-07,Não



LINHAS FORMATADAS PARA A TABELA LATEX
Original Theme & $0.286 \pm 0.113$ &
$0.775 \pm 0.078$ \\

Intermediate Theme & $0.279 \pm 0.140$ &
$0.663 \pm 0.097$ \\

General Theme & $0.141 \pm 0.062$ &
$0.391 \pm 0.078$ \\



1715

## Análise de correlação (tamanho dos modelos abertos vs desempenho)

In [23]:
df_all["size"] = df_all.embedding.apply(get_model_size)

In [24]:
embedding_cols_abertos = [
    'embedding__codefuse_ai__F2LLM_v2_14B__texto',
       'embedding__codefuse_ai__F2LLM_v2_14B__texto_preprocessado',
       'embedding__codefuse_ai__F2LLM_v2_14B__texto_preprocessado_sem_justificativa',
       'embedding__Octen__Octen_Embedding_8B__texto',
       'embedding__Octen__Octen_Embedding_8B__texto_preprocessado',
       'embedding__Octen__Octen_Embedding_8B__texto_preprocessado_sem_justificativa',
       'embedding__Qwen__Qwen3_Embedding_8B__texto',
       'embedding__Qwen__Qwen3_Embedding_8B__texto_preprocessado',
       'embedding__Qwen__Qwen3_Embedding_8B__texto_preprocessado_sem_justificativa',
       'embedding__nvidia__llama_embed_nemotron_8b__texto',
       'embedding__nvidia__llama_embed_nemotron_8b__texto_preprocessado',
       'embedding__nvidia__llama_embed_nemotron_8b__texto_preprocessado_sem_justificativa',
       'embedding__jinaai__jina_embeddings_v5_text_small__text_matching__texto',
       'embedding__jinaai__jina_embeddings_v5_text_small__text_matching__texto_preprocessado',
       'embedding__jinaai__jina_embeddings_v5_text_small__text_matching__texto_preprocessado_sem_justificativa',
       'embedding__joaorobson__harrier_oss_v1_27b__texto',
       'embedding__joaorobson__harrier_oss_v1_27b__texto_preprocessado',
       'embedding__joaorobson__harrier_oss_v1_27b__texto_preprocessado_sem_justificativa',
       'embedding__joaorobson__KaLM_Embedding_Gemma3_12B_2511__texto',
       'embedding__joaorobson__KaLM_Embedding_Gemma3_12B_2511__texto_preprocessado',
       'embedding__joaorobson__KaLM_Embedding_Gemma3_12B_2511__texto_preprocessado_sem_justificativa',
       'embedding__PORTULAN__serafim_335m_portuguese_pt_sentence_encoder__trunc128__texto',
       'embedding__PORTULAN__serafim_335m_portuguese_pt_sentence_encoder__trunc128__texto_preprocessado',
       'embedding__PORTULAN__serafim_335m_portuguese_pt_sentence_encoder__trunc128__texto_preprocessado_sem_justificativa',
       'embedding__PORTULAN__serafim_335m_portuguese_pt_sentence_encoder__meanpool128__texto',
       'embedding__PORTULAN__serafim_335m_portuguese_pt_sentence_encoder__meanpool128__texto_preprocessado',
       'embedding__PORTULAN__serafim_335m_portuguese_pt_sentence_encoder__meanpool128__texto_preprocessado_sem_justificativa',
       'embedding__PORTULAN__serafim_335m_portuguese_pt_sentence_encoder_ir__trunc128__texto',
       'embedding__PORTULAN__serafim_335m_portuguese_pt_sentence_encoder_ir__trunc128__texto_preprocessado',
       'embedding__PORTULAN__serafim_335m_portuguese_pt_sentence_encoder_ir__trunc128__texto_preprocessado_sem_justificativa',
       'embedding__PORTULAN__serafim_335m_portuguese_pt_sentence_encoder_ir__meanpool128__texto',
       'embedding__PORTULAN__serafim_335m_portuguese_pt_sentence_encoder_ir__meanpool128__texto_preprocessado',
       'embedding__PORTULAN__serafim_335m_portuguese_pt_sentence_encoder_ir__meanpool128__texto_preprocessado_sem_justificativa',
       'embedding__josedossantos__bertimbau_tuned__trunc512__texto',
       'embedding__josedossantos__bertimbau_tuned__trunc512__texto_preprocessado',
       'embedding__josedossantos__bertimbau_tuned__trunc512__texto_preprocessado_sem_justificativa',
       'embedding__josedossantos__bertimbau_tuned__meanpool512__texto',
       'embedding__josedossantos__bertimbau_tuned__meanpool512__texto_preprocessado',
       'embedding__josedossantos__bertimbau_tuned__meanpool512__texto_preprocessado_sem_justificativa',
       'embedding__ICT_TIME_and_Querit__ICT_TIME_and_Querit_embedding_v1__texto',
       'embedding__ICT_TIME_and_Querit__ICT_TIME_and_Querit_embedding_v1__texto_preprocessado',
       'embedding__ICT_TIME_and_Querit__ICT_TIME_and_Querit_embedding_v1__texto_preprocessado_sem_justificativa']

df_abertos = df_all[(df_all.tipo_texto == "texto_preprocessado") & (df_all.nivel == "tema") & (df_all.embedding.isin(embedding_cols_abertos))].copy()

In [25]:
from scipy.stats import spearmanr
import pandas as pd

metrics = ["ari", "v_measure"]

results = []

for materia in df_abertos["materia"].unique():
    sub = df_abertos[df_abertos["materia"] == materia]
    
    for metric in metrics:
        rho, p = spearmanr(sub["size"], sub[metric])
        
        results.append({
            "materia": materia,
            "metric": metric,
            "spearman_rho": rho,
            "p_value": p
        })

df_corr = pd.DataFrame(results)
df_corr

,materia,metric,spearman_rho,p_value
0,PEC_6_2019,ari,0.815973,0.000372
1,PEC_6_2019,v_measure,0.841257,0.000163
2,MPV_612_2013,ari,0.836660,0.000191
3,MPV_612_2013,v_measure,0.818272,0.000347
4,PLP_68_2024,ari,0.809078,0.000457
5,PLP_68_2024,v_measure,0.832063,0.000223


In [43]:
import numpy as np
import pandas as pd
from scipy.stats import linregress

# =====================================================
# 1. MAPEAMENTO DE TAMANHOS E DEFINIÇÃO DAS FAIXAS (TIERS)
# =====================================================

MODEL_SIZE = {
    "F2LLM_v2_14B": 14.0,
    "Octen_Embedding_8B": 8.0,
    "Qwen3_Embedding_8B": 8.0,
    "llama_embed_nemotron_8b": 8.0,
    "harrier_oss_v1_27b": 27.0,
    "KaLM_Embedding_Gemma3_12B_2511": 12.0,
    "jina_embeddings_v5_text_small": 0.6,
    "serafim_335m_portuguese_pt_sentence_encoder": 0.335,
    "serafim_335m_portuguese_pt_sentence_encoder_ir": 0.335,
    "bertimbau_tuned": 0.335,
    "ICT_TIME_and_Querit_embedding_v1": 4.0
}

def get_model_size(name):
    for model, size in MODEL_SIZE.items():
        if model in name:
            return size
    return np.nan

def assign_tier(size):
    """Categoriza o modelo por faixa de capacidade em parâmetros."""
    if size < 1.0:
        return "1. Small / Specialized (< 1B)"
    elif size <= 8.0:
        return "2. Mid-scale (4B - 8B)"
    elif size > 8.0:
        return "3. Large Open (12B - 27B)"
    return np.nan

# =====================================================
# 2. PREPARAÇÃO DOS DADOS COM FILTRO (nivel == 'tema')
# =====================================================

df_analise = df_abertos.copy()

# Filtra estritamente o nível original/específico
if "nivel" in df_analise.columns:
    df_analise = df_analise[df_analise["nivel"] == "tema"].copy()

# Mapeia e filtra apenas modelos abertos com tamanho conhecido
df_analise["size_b"] = df_analise["embedding"].apply(get_model_size)
df_analise = df_analise.dropna(subset=["size_b"]).copy()

df_analise["tier"] = df_analise["size_b"].apply(assign_tier)
df_analise["log_size"] = np.log10(df_analise["size_b"])

metrics = ["ari", "v_measure"]

# =====================================================
# 3. TABELA 1: MÉDIAS E DESVIOS POR FAIXA (TIERED SUMMARY)
# =====================================================

tier_summary = (
    df_analise
    .groupby(["materia", "tier"])[metrics]
    .agg(["mean", "std"])
    .round(4)
)

print("=" * 70)
print("DESEMPENHO MÉDIO POR FAIXA DE PARÂMETROS (TIERS) - NÍVEL TEMA")
print("=" * 70)
display(tier_summary)

# =====================================================
# 4. TABELA 2: REGRESSÃO LOG-LINEAR (SCALING LAWS)
# =====================================================

reg_results = []

for materia in df_analise["materia"].unique():
    sub = df_analise[df_analise["materia"] == materia]
    
    for metric in metrics:
        slope, intercept, r_value, p_value, std_err = linregress(sub["log_size"], sub[metric])
        
        reg_results.append({
            "materia": materia,
            "metric": metric,
            "slope_beta (ganho por 10x)": round(slope, 4),
            "r_squared (R²)": round(r_value**2, 4),
            "p_value": round(p_value, 4)
        })

df_reg = pd.DataFrame(reg_results)

print("\n" + "=" * 70)
print("REGRESSÃO LOG-LINEAR: MÉTRICA ~ log10(SIZE) - NÍVEL TEMA")
print("=" * 70)
display(df_reg)

DESEMPENHO MÉDIO POR FAIXA DE PARÂMETROS (TIERS) - NÍVEL TEMA


ari         v_measure        
                                              mean     std      mean     std
materia      tier                                                           
MPV_612_2013 1. Small / Specialized (< 1B)  0.1196  0.0137    0.6259  0.0167
             2. Mid-scale (4B - 8B)         0.2175  0.0383    0.7053  0.0135
             3. Large Open (12B - 27B)      0.2468  0.0352    0.7046  0.0129
PEC_6_2019   1. Small / Specialized (< 1B)  0.2221  0.0807    0.5961  0.0659
             2. Mid-scale (4B - 8B)         0.4037  0.0486    0.7350  0.0382
             3. Large Open (12B - 27B)      0.4207  0.1269    0.7323  0.0693
PLP_68_2024  1. Small / Specialized (< 1B)  0.1652  0.1154    0.6980  0.0959
             2. Mid-scale (4B - 8B)         0.3658  0.0200    0.8227  0.0067
             3. Large Open (12B - 27B)      0.3495  0.0493    0.8164  0.0189


REGRESSÃO LOG-LINEAR: MÉTRICA ~ log10(SIZE) - NÍVEL TEMA


,materia,metric,slope_beta (ganho por 10x),r_squared (R²),p_value
0,PEC_6_2019,ari,0.1331,0.6653,0.0004
1,PEC_6_2019,v_measure,0.0958,0.6598,0.0004
2,MPV_612_2013,ari,0.0747,0.8180,0.0000
3,MPV_612_2013,v_measure,0.0525,0.8621,0.0000
4,PLP_68_2024,ari,0.1339,0.6300,0.0007
5,PLP_68_2024,v_measure,0.0839,0.4983,0.0048


## Resultados por modelo

In [41]:
(
    df_all.query("nivel == 'tema'")
    .sort_values(
        ["materia", "tipo_texto", "ari"],
        ascending=[True, True, False]
    )
    .groupby(["materia", "tipo_texto"], group_keys=False)
    .head(5)
    .reset_index(drop=True)
)

,nivel,embedding,modelo,ari,v_measure,materia,tipo_texto,size
0,tema,embedding__Qwen__Qwen3_Embedding_8B__texto,agglomerative,0.236870,0.683678,MPV_612_2013,texto,8.0
1,tema,embedding__codefuse_ai__F2LLM_v2_14B__texto,agglomerative,0.214929,0.663390,MPV_612_2013,texto,14.0
2,tema,embedding__joaorobson__harrier_oss_v1_27b__texto,agglomerative,0.205328,0.691960,MPV_612_2013,texto,27.0
3,tema,embedding__joaorobson__KaLM_Embedding_Gemma3_1...,agglomerative,0.191741,0.699121,MPV_612_2013,texto,12.0
4,tema,embedding__Octen__Octen_Embedding_8B__texto,agglomerative,0.172873,0.691535,MPV_612_2013,texto,8.0
5,tema,embedding__joaorobson__harrier_oss_v1_27b__tex...,agglomerative,0.277524,0.714753,MPV_612_2013,texto_preprocessado,27.0
6,tema,embedding__ICT_TIME_and_Querit__ICT_TIME_and_Q...,agglomerative,0.255449,0.719338,MPV_612_2013,texto_preprocessado,4.0
7,tema,embedding__joaorobson__KaLM_Embedding_Gemma3_1...,agglomerative,0.254482,0.708883,MPV_612_2013,texto_preprocessado,12.0
8,tema,embedding__Octen__Octen_Embedding_8B__texto_pr...,agglomerative,0.241417,0.710784,MPV_612_2013,texto_preprocessado,8.0
9,tema,bm25l_preprocess__texto_preprocessado,agglomerative,0.224040,0.711967,MPV_612_2013,texto_preprocessado,NaN


In [42]:
(
    df_all.query("nivel == 'tema'")
    .sort_values(
        ["materia", "tipo_texto", "v_measure"],
        ascending=[True, True, False]
    )
    .groupby(["materia", "tipo_texto"], group_keys=False)
    .head(5)
    .reset_index(drop=True)
)

,nivel,embedding,modelo,ari,v_measure,materia,tipo_texto,size
0,tema,embedding__joaorobson__KaLM_Embedding_Gemma3_1...,agglomerative,0.191741,0.699121,MPV_612_2013,texto,12.0
1,tema,embedding__joaorobson__harrier_oss_v1_27b__texto,agglomerative,0.205328,0.691960,MPV_612_2013,texto,27.0
2,tema,embedding__Octen__Octen_Embedding_8B__texto,agglomerative,0.172873,0.691535,MPV_612_2013,texto,8.0
3,tema,embedding__Qwen__Qwen3_Embedding_8B__texto,agglomerative,0.236870,0.683678,MPV_612_2013,texto,8.0
4,tema,embedding__codefuse_ai__F2LLM_v2_14B__texto,agglomerative,0.214929,0.663390,MPV_612_2013,texto,14.0
5,tema,embedding__ICT_TIME_and_Querit__ICT_TIME_and_Q...,agglomerative,0.255449,0.719338,MPV_612_2013,texto_preprocessado,4.0
6,tema,embedding__joaorobson__harrier_oss_v1_27b__tex...,agglomerative,0.277524,0.714753,MPV_612_2013,texto_preprocessado,27.0
7,tema,bm25l_preprocess__texto_preprocessado,agglomerative,0.224040,0.711967,MPV_612_2013,texto_preprocessado,NaN
8,tema,embedding__Octen__Octen_Embedding_8B__texto_pr...,agglomerative,0.241417,0.710784,MPV_612_2013,texto_preprocessado,8.0
9,tema,embedding__joaorobson__KaLM_Embedding_Gemma3_1...,agglomerative,0.254482,0.708883,MPV_612_2013,texto_preprocessado,12.0


In [28]:
import pandas as pd

metricas = ["ari", "v_measure"]

ranks = []

for metrica in metricas:
    r = (
        df_all[(df_all.tipo_texto == "texto_preprocessado") & (df_all.nivel == "tema")].groupby("materia")
          .apply(lambda x:
                 x.assign(
                     rank=x[metrica].rank(
                         ascending=False,
                         method="average"
                     )
                 )[["embedding", "materia", "rank"]]
          )
          .reset_index(drop=True)
    )
    r["metrica"] = metrica
    ranks.append(r)

ranks = pd.concat(ranks)

ranking_medio = (
    ranks.groupby("embedding")
         .agg(
             ranking_medio=("rank","mean"),
             desvio=("rank","std"),
             melhor=("rank","min"),
             pior=("rank","max")
         )
         .sort_values("ranking_medio")
)

ranking_medio

C:\Users\3675-Robson\AppData\Local\Temp\ipykernel_18032\3623029288.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x:
C:\Users\3675-Robson\AppData\Local\Temp\ipykernel_18032\3623029288.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x:


,ranking_medio,desvio,melhor,pior
embedding,,,,
embedding__joaorobson__harrier_oss_v1_27b__texto_preprocessado,2.333333,0.816497,1.0,3.0
embedding__Octen__Octen_Embedding_8B__texto_preprocessado,3.000000,1.549193,1.0,4.0
embedding__codefuse_ai__F2LLM_v2_14B__texto_preprocessado,3.666667,3.076795,1.0,8.0
embedding__ICT_TIME_and_Querit__ICT_TIME_and_Querit_embedding_v1__texto_preprocessado,3.666667,1.751190,1.0,5.0
embedding__Qwen__Qwen3_Embedding_8B__texto_preprocessado,6.166667,3.868678,2.0,11.0
embedding__openai__text_embedding_3_large__trunc8192__texto_preprocessado,6.833333,1.602082,6.0,10.0
embedding__nvidia__llama_embed_nemotron_8b__texto_preprocessado,7.333333,0.816497,6.0,8.0
bm25l_preprocess__texto_preprocessado,7.500000,4.086563,3.0,13.0
embedding__joaorobson__KaLM_Embedding_Gemma3_12B_2511__texto_preprocessado,8.166667,3.371449,3.0,11.0
